# 05. Elastic parameters: Poisson's ratio and Young's modulus

**Thesis:** Machine-Learning Prediction of Shear-Wave Velocity in the Dampier Sub-basin: Accuracy, Errors and Applications (D. Aldosaimani, Curtin University, 2026)

**What this notebook does:** Repeats the Lasso LOWO prediction, keeps the prediction at every depth, and calculates Poisson's ratio (σ) and Young's modulus (E) from the measured and the predicted Vs, with their accuracy in each well.

**Used in:** Section 4.5; Appendix A, Table A3 and Figure A4

**Needs:** `DATA_DIR` and `LAS_DIR` (set in the Settings cell below)

**Writes:** σ and E at every depth; accuracy of σ and E; error by formation

## 1. Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import lasio
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# All plots are saved into this folder (created if it doesn't exist yet), at 300 dpi
PLOTS_DIR = Path('plots')
PLOTS_DIR.mkdir(exist_ok=True)
PLOT_DPI = 300

## Settings

In [ ]:
# ---- Settings: set these paths before running ----
DATA_DIR    = Path("../data/processed")   # the eight *_with_formation.csv files (well logs with formation names)
LAS_DIR     = Path("../data/las")         # the eight *_LQC.las files from NOPIMS (project ENO0810814A)
RESULTS_DIR = Path("../results")          # results tables are written here
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load and combine wells

Same data folder as the regression notebook.

In [ ]:
from pathlib import Path
import pandas as pd
data_dir = DATA_DIR

well_logs = {}
for csv_file in sorted(data_dir.glob("[!.]*.csv")):
    well_name = csv_file.stem
    df = pd.read_csv(csv_file)
    df.insert(0, "Well Name", well_name)
    well_logs[well_name] = df
    print(f"{well_name}: {len(df)} samples")

data_df = pd.concat(well_logs.values(), ignore_index=True)
print(f"\nLoaded {len(well_logs)} wells, {len(data_df)} total samples")

## 2b. Data cleaning (final)

Same two cleaning cells as the Regression notebook (updated_8), so every model and every result uses exactly the same rows:

1. **Remove impossible Vp:** readings faster than the fastest Vp the local minerals could have at zero porosity (Voigt bound of quartz, calcite and illite, illite given quartz properties; Mavko et al., 2009).
2. **Keep only real, unchanged measured Vs** in the 5 measured wells: `VS_ORG` exists in the LAS file and `VS_EDITED_V2` is within 2% of it.

Washouts are kept (tested in the Regression notebook), and no rule based on Vs itself is used. The 3 modelled-only wells only go through step 1.


In [ ]:
# --- Remove physically impossible Vp (faster than the rock's own minerals) ---
# Upper limit at each depth = stiffest possible (Voigt) mix of its minerals at zero porosity.
# Clay (illite) is given the stiffness of quartz. Real clay is softer, so this limit is
# deliberately too high: it can only remove readings that no real rock could produce.
# Mineral properties from Mavko et al. (2009): K and mu in GPa, rho in g/cc
QUARTZ  = dict(K=37.0, mu=44.0, rho=2.65)     # solid quartz:  Vp ~ 6010 m/s
CALCITE = dict(K=76.8, mu=32.0, rho=2.71)     # solid calcite: Vp ~ 6640 m/s
REMOVE_IMPOSSIBLE_VP = True

def p_modulus(mineral):                        # P-wave modulus M = K + 4/3 mu
    return mineral['K'] + 4 / 3 * mineral['mu']

solid = data_df[['QUARTZ_COMBINER', 'CALCITE_COMBINER', 'ILLITE_COMBINER']].sum(axis=1)
f_calcite = data_df['CALCITE_COMBINER'] / solid      # calcite fraction of the solid rock
f_other = 1 - f_calcite                               # quartz + illite (both given quartz properties)

M   = f_other * p_modulus(QUARTZ) + f_calcite * p_modulus(CALCITE)      # GPa
rho = f_other * QUARTZ['rho'] + f_calcite * CALCITE['rho']               # g/cc
data_df['VP_LIMIT'] = np.sqrt(M / rho) * 1000                             # km/s -> m/s
data_df['vp_impossible'] = data_df['VP_EDITED'] > data_df['VP_LIMIT']

bad = data_df[data_df['vp_impossible']]
print(f"Depths with Vp faster than their minerals allow: {len(bad)}")
display(bad[['Well Name', 'MD', 'Formation', 'VP_EDITED', 'VP_LIMIT']].round(1))

if REMOVE_IMPOSSIBLE_VP:
    data_df = data_df[~data_df['vp_impossible']].reset_index(drop=True)
    print(f"Removed. Rows left: {len(data_df)}")


In [ ]:
# --- Keep only real measured Vs (compare VS_EDITED_V2 with VS_ORG from the LAS files) ---

MEASURED_ONLY = True   # remove depths where Vs is not a real measurement
DROP_EDITED   = True   # also remove depths where SLB changed the measured Vs by more than EDIT_TOL
EDIT_TOL      = 0.02   # 2%, same threshold as the checks in Part 3

measured_names = ['Egret_2', 'Lambert_8B', 'Nereus_1', 'Persephone_1', 'Seraph_1']
data_df['VS_ORG'] = np.nan             # modelled-only wells stay empty

for w in measured_names:
    d = lasio.read(list(LAS_DIR.rglob(f"{w}_LQC.las"))[0]).df().reset_index()
    d.columns = d.columns.str.upper()
    d = d.rename(columns={d.columns[0]: 'MD'})
    d = d[['MD', 'VS_ORG']].sort_values('MD')

    idx = data_df['Well Name'].str.contains(w)
    sub = data_df.loc[idx, ['MD']].reset_index().sort_values('MD')
    merged = pd.merge_asof(sub, d, on='MD', direction='nearest', tolerance=0.1)
    data_df.loc[merged['index'], 'VS_ORG'] = merged['VS_ORG'].values

has_org = data_df['VS_ORG'].notna()
edited = has_org & ((data_df['VS_EDITED_V2'] - data_df['VS_ORG']).abs() / data_df['VS_ORG'] > EDIT_TOL)
data_df['VS_MEASURED'] = (has_org & ~edited) if DROP_EDITED else has_org

# Summary per measured well
in_measured_well = data_df['Well Name'].str.contains('|'.join(measured_names))
names = data_df.loc[in_measured_well, 'Well Name']
summary = pd.DataFrame({
    'rows': names.value_counts().sort_index(),
    '% real, unchanged': (100 * (has_org & ~edited)[in_measured_well].groupby(names).mean()).round(1),
    '% real, edited by SLB': (100 * edited[in_measured_well].groupby(names).mean()).round(1),
    '% modelled (no VS_ORG)': (100 * (~has_org)[in_measured_well].groupby(names).mean()).round(1),
})
print(summary)

if MEASURED_ONLY:
    data_df = data_df[~in_measured_well | data_df['VS_MEASURED']].reset_index(drop=True)
    print("\nRows kept per well:")
    print(data_df.groupby('Well Name').size())

    # Check: every row left in the 5 measured wells is a real measurement
    in_measured_well = data_df['Well Name'].str.contains('|'.join(measured_names))
    assert data_df.loc[in_measured_well, 'VS_MEASURED'].all()
    print("\nCheck passed: all rows left in the 5 measured wells are real measured Vs")


## 3. Feature set

Identical to the regression notebook: 5 features (RESD_LOG, no VCL_GEO_QEPP). Each LOWO fold trains on the other 4 measured wells and predicts the 5th.

In [ ]:
measured_wells = [
    'Egret_2_with_formation',
    'Lambert_8B_with_formation',
    'Nereus_1_with_formation',
    'Persephone_1_with_formation',
    'Seraph_1_with_formation',
]

data_df['RESD_LOG'] = np.log10(data_df['RESD_EDITED'])

features_reg = ['GR_EDITED', 'RHOB_EDITED', 'NPHI_EDITED', 'RESD_LOG', 'VP_EDITED']
target_reg = 'VS_EDITED_V2'

## 4. Wells to look at

All 5 measured wells now (pilot started with just Persephone_1 and Seraph_1 — this extends the same pipeline to Egret_2, Lambert_8B and Nereus_1 too).

In [ ]:
WELLS_OF_INTEREST = list(measured_wells)

## 5. Method — Lasso alpha selection (LOWO)

Same sweep as the regression notebook's Section 8: for each candidate alpha, rotate through the 5 measured wells (train on 4, predict the 5th), and keep the alpha with the lowest mean LOWO RMSE.

In [ ]:
alphas = np.logspace(-3, 4, 30)
sweep_results = []

for alpha in alphas:
    fold_rmse = []
    for val_well in measured_wells:
        train_df = data_df[(data_df['Well Name'] != val_well) & (data_df['Well Name'].isin(measured_wells))]
        val_df = data_df[data_df['Well Name'] == val_well]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(train_df[features_reg])
        X_val = scaler.transform(val_df[features_reg])

        model = Lasso(alpha=alpha, max_iter=20000, tol=1e-3)
        model.fit(X_train, train_df[target_reg])
        pred = model.predict(X_val)
        fold_rmse.append(np.sqrt(mean_squared_error(val_df[target_reg], pred)))

    sweep_results.append({'alpha': alpha, 'mean_rmse': np.mean(fold_rmse)})

sweep_df = pd.DataFrame(sweep_results)
best_alpha = sweep_df.loc[sweep_df['mean_rmse'].idxmin(), 'alpha']
print(f"Best alpha by mean LOWO RMSE: {best_alpha:.4f}")

## 6. Method — final Lasso LOWO, keep per-depth predictions

Same fold structure, fixed at best_alpha, keeping every row's predicted Vs (not just the summary R²/RMSE) — that's what the physics step needs.

In [ ]:
results = []
per_depth_frames = []

for val_well in measured_wells:
    train_df = data_df[(data_df['Well Name'] != val_well) & (data_df['Well Name'].isin(measured_wells))]
    val_df = data_df[data_df['Well Name'] == val_well].copy()

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[features_reg])
    X_val = scaler.transform(val_df[features_reg])

    model = Lasso(alpha=best_alpha, max_iter=20000, tol=1e-3)
    model.fit(X_train, train_df[target_reg])
    val_df['VS_PRED_LASSO'] = model.predict(X_val)

    results.append({
        'Validation Well': val_well,
        'R2': r2_score(val_df[target_reg], val_df['VS_PRED_LASSO']),
        'RMSE': np.sqrt(mean_squared_error(val_df[target_reg], val_df['VS_PRED_LASSO'])),
    })
    per_depth_frames.append(val_df[['Well Name', 'MD', 'VP_EDITED', 'RHOB_EDITED', target_reg, 'VS_PRED_LASSO']])

results_lasso = pd.DataFrame(results)
per_depth_df = pd.concat(per_depth_frames, ignore_index=True).rename(columns={target_reg: 'VS_ACTUAL'})

print(f"Per-well LOWO metrics (Lasso, alpha={best_alpha:.4f}):")
results_lasso

## 7. Application — Poisson's ratio and Young's modulus

σ = (Vp² − 2Vs²) / (2(Vp² − Vs²));  E = 2ρVs²(1+σ), with ρ converted from g/cc to kg/m³ and E reported in GPa. Computed from both the predicted Vs (the actual result) and the measured Vs (for comparison only).

In [ ]:
def compute_sigma(vp, vs):
    return (vp**2 - 2*vs**2) / (2*(vp**2 - vs**2))

def compute_E_GPa(rhob_gcc, vs, sigma):
    rho_kgm3 = rhob_gcc * 1000.0
    return 2 * rho_kgm3 * vs**2 * (1 + sigma) / 1e9

per_depth_df['sigma'] = compute_sigma(per_depth_df['VP_EDITED'], per_depth_df['VS_PRED_LASSO'])
per_depth_df['E_GPa'] = compute_E_GPa(per_depth_df['RHOB_EDITED'], per_depth_df['VS_PRED_LASSO'], per_depth_df['sigma'])

per_depth_df['sigma_actual'] = compute_sigma(per_depth_df['VP_EDITED'], per_depth_df['VS_ACTUAL'])
per_depth_df['E_GPa_actual'] = compute_E_GPa(per_depth_df['RHOB_EDITED'], per_depth_df['VS_ACTUAL'], per_depth_df['sigma_actual'])

per_depth_df[['Well Name', 'sigma', 'E_GPa']].groupby('Well Name').agg(['mean', 'min', 'max'])

## 8. Save results

In [ ]:
results_of_interest = per_depth_df[per_depth_df['Well Name'].isin(WELLS_OF_INTEREST)]
results_of_interest.to_csv(RESULTS_DIR / 'sigma_E_results.csv', index=False)
print('Saved sigma_E_results.csv, shape:', results_of_interest.shape)

## 9. Depth plot — σ and E vs depth

Black = from measured Vs, crimson = from predicted Vs. No uncertainty band in this version. Gaps in the curves are depths removed in Section 2b (no real measured Vs). One separate figure per well, each saved to its own PNG in `plots/` at 300 dpi.

In [ ]:
def with_gaps(g, max_step=2.0):
    """Insert an empty row wherever two depths are more than max_step m apart, so the
    line stops at removed (modelled-Vs) intervals instead of drawing a straight line across them."""
    g = g.sort_values('MD').reset_index(drop=True)
    gap = g['MD'].diff() > max_step
    if not gap.any():
        return g
    blanks = g.loc[gap, ['MD']] - 0.001          # just above the next real sample, all other columns empty
    return pd.concat([g, blanks]).sort_values('MD').reset_index(drop=True)

for well in WELLS_OF_INTEREST:
    g = with_gaps(per_depth_df[per_depth_df['Well Name'] == well])
    depth = g['MD']
    well_label = well.replace('_with_formation', '')

    fig, axes = plt.subplots(1, 2, figsize=(11, 7))

    ax = axes[0]
    ax.plot(g['sigma_actual'], depth, color='black', lw=0.8, label='σ from measured Vs')
    ax.plot(g['sigma'], depth, color='crimson', lw=0.8, label='σ from predicted Vs')
    ax.invert_yaxis()
    ax.set_xlabel("Poisson's ratio σ")
    ax.set_ylabel('MD (m)')
    ax.set_title(well_label)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[1]
    ax.plot(g['E_GPa_actual'], depth, color='black', lw=0.8, label='E from measured Vs')
    ax.plot(g['E_GPa'], depth, color='crimson', lw=0.8, label='E from predicted Vs')
    ax.invert_yaxis()
    ax.set_xlabel("Young's modulus E (GPa)")
    ax.set_ylabel('MD (m)')
    ax.set_title(well_label)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

    fig.suptitle(f'σ and E vs depth — {well_label}', y=1.02)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f'depth_plot_{well_label}.png', dpi=PLOT_DPI, bbox_inches='tight')
    plt.show()

## 10. E–σ crossplot

Every depth's E plotted against σ, colored by depth. Gray = from measured Vs, colored = from predicted Vs. One separate figure per well, each saved to its own PNG in `plots/` at 300 dpi.

In [ ]:
blue_ramp = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#1c5cab', '#104281', '#0d366b']
blue_cmap = LinearSegmentedColormap.from_list('seq_blue', blue_ramp)

for well in WELLS_OF_INTEREST:
    g = per_depth_df[per_depth_df['Well Name'] == well]
    well_label = well.replace('_with_formation', '')

    fig, ax = plt.subplots(figsize=(6.5, 5.5))

    ax.scatter(g['sigma_actual'], g['E_GPa_actual'], s=6, color='#9a9a94', alpha=0.35,
               label='from measured Vs', zorder=1)
    sc = ax.scatter(g['sigma'], g['E_GPa'], s=8, c=g['MD'], cmap=blue_cmap,
                     alpha=0.85, zorder=2, label='from predicted Vs')

    ax.set_xlabel("Poisson's ratio σ")
    ax.set_ylabel("Young's modulus E (GPa)")
    ax.set_title(f'E–σ crossplot (predicted Vs, colored by depth) — {well_label}')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='upper right')

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label('MD (m)')
    cbar.ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f'crossplot_{well_label}.png', dpi=PLOT_DPI, bbox_inches='tight')
    plt.show()

## 11. Accuracy metrics — R², RMSE, MAE for σ and E (not just Vs)

Section 6 already gives R²/RMSE for Vs itself (how good the Lasso model is). But turning Vs into σ and E is an extra step, and that step can hide problems — e.g. Persephone_1 looked fine on the σ depth plot, but its σ R² actually came out close to 0. So here we compute the same accuracy metrics (R², RMSE, MAE), but for σ and E themselves, comparing 'from predicted Vs' against 'from measured Vs' at every depth, for every well — plus MAE for Vs, which Section 6 didn't include either.

RMSE is also expressed as a percentage of each parameter's own range in that well, because Vs (m/s), σ (roughly 0–0.4) and E (tens of GPa) are on completely different scales — RMSE alone can't be compared across them, but RMSE-as-%-of-range can.

In [ ]:
PARAMS = [
    ('Vs', 'VS_ACTUAL', 'VS_PRED_LASSO'),
    ("Poisson's ratio (sigma)", 'sigma_actual', 'sigma'),
    ("Young's modulus (E)", 'E_GPa_actual', 'E_GPa'),
]

metrics_rows = []
for well in WELLS_OF_INTEREST:
    g = per_depth_df[per_depth_df['Well Name'] == well]
    well_label = well.replace('_with_formation', '')

    for param_name, actual_col, pred_col in PARAMS:
        actual = g[actual_col]
        pred = g[pred_col]
        r2 = r2_score(actual, pred)
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        value_range = actual.max() - actual.min()
        rmse_pct_range = 100 * rmse / value_range

        metrics_rows.append({
            'Well': well_label,
            'Parameter': param_name,
            'R2': r2,
            'RMSE': rmse,
            'MAE': mae,
            'RMSE_pct_range': rmse_pct_range,
        })

accuracy_metrics_df = pd.DataFrame(metrics_rows)
accuracy_metrics_df.to_csv(RESULTS_DIR / 'accuracy_metrics_sigma_E.csv', index=False)
print('Saved accuracy_metrics_sigma_E.csv, shape:', accuracy_metrics_df.shape)
accuracy_metrics_df.round(4)

## 12. Visualize accuracy — R² and RMSE% across wells and parameters

Grouped bar chart, one bar per parameter (Vs, σ, E) per well. Left panel: R² (higher is better; 0 means 'no better than guessing the average', negative means worse than that). Right panel: RMSE as a percentage of each parameter's own range (lower is better) — this is what makes σ and E comparable to each other despite being on totally different scales.

In [ ]:
param_names = [p[0] for p in PARAMS]
param_colors = {
    'Vs': '#2a78d6',
    "Poisson's ratio (sigma)": '#eb6834',
    "Young's modulus (E)": '#1baf7a',
}

wells_order = [w.replace('_with_formation', '') for w in WELLS_OF_INTEREST]
x = np.arange(len(wells_order))
bar_width = 0.26

def style_axis(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#c3c2b7')
    ax.spines['bottom'].set_color('#c3c2b7')
    ax.tick_params(colors='#52514e')
    ax.set_axisbelow(True)
    ax.grid(axis='y', color='#e1e0d9', linewidth=1, zorder=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.2))

# --- Panel A: R2 ---
ax = axes[0]
for i, param_name in enumerate(param_names):
    sub = accuracy_metrics_df[accuracy_metrics_df['Parameter'] == param_name].set_index('Well')
    vals = [sub.loc[w, 'R2'] for w in wells_order]
    bars = ax.bar(x + (i - 1) * bar_width, vals, width=bar_width * 0.92,
                   color=param_colors[param_name], label=param_name,
                   edgecolor='white', linewidth=1, zorder=3)
    for rect, v in zip(bars, vals):
        va, offset = ('bottom', 0.02) if v >= 0 else ('top', -0.02)
        ax.text(rect.get_x() + rect.get_width() / 2, v + offset, f'{v:.2f}',
                ha='center', va=va, fontsize=7.5, color='#0b0b0b')

ax.axhline(0, color='#898781', lw=1, zorder=2)
ax.set_xticks(x)
ax.set_xticklabels(wells_order, rotation=25, ha='right', fontsize=9.5)
ax.set_ylabel('R²', fontsize=10)
ax.set_title('R²', fontsize=10.5, pad=10)
ax.set_ylim(accuracy_metrics_df['R2'].min() - 0.15, 1.08)
style_axis(ax)

# --- Panel B: RMSE as % of range ---
ax = axes[1]
for i, param_name in enumerate(param_names):
    sub = accuracy_metrics_df[accuracy_metrics_df['Parameter'] == param_name].set_index('Well')
    vals = [sub.loc[w, 'RMSE_pct_range'] for w in wells_order]
    bars = ax.bar(x + (i - 1) * bar_width, vals, width=bar_width * 0.92,
                   color=param_colors[param_name], label=param_name,
                   edgecolor='white', linewidth=1, zorder=3)
    for rect, v in zip(bars, vals):
        ax.text(rect.get_x() + rect.get_width() / 2, v + 0.3, f'{v:.1f}',
                ha='center', va='bottom', fontsize=7.5, color='#0b0b0b')

ax.set_xticks(x)
ax.set_xticklabels(wells_order, rotation=25, ha='right', fontsize=9.5)
ax.set_ylabel('RMSE as % of parameter range', fontsize=10)
ax.set_title('RMSE (relative)', fontsize=10.5, pad=10)
ax.set_ylim(0, accuracy_metrics_df['RMSE_pct_range'].max() * 1.2)
style_axis(ax)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.06), ncol=3,
           frameon=False, fontsize=10)
fig.suptitle("Vs, Poisson's ratio (σ) and Young's modulus (E) — accuracy by well",
             y=1.15, fontsize=12.5)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'accuracy_metrics_comparison.png', dpi=PLOT_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# --- Interpretation: where and why do predictions fail in one well? ---
# Change WELL to look at another well (e.g. 'Egret_2_with_formation')
WELL = 'Persephone_1_with_formation'
name = WELL.replace('_with_formation', '')

extra = ['Well Name', 'MD', 'Formation', 'GR_EDITED', 'CALCITE_COMBINER', 'CALI_EDITED']
p = (per_depth_df[per_depth_df['Well Name'] == WELL]
     .merge(data_df[extra], on=['Well Name', 'MD'], how='left')
     .sort_values('MD').reset_index(drop=True))

# Gas saturation from the LAS file (same method as estimate_sat_from_combiner.py)
las = lasio.read(list(LAS_DIR.rglob(f"{name}_LQC.las"))[0]).df().reset_index()
las.columns = las.columns.str.upper()
las = las.rename(columns={las.columns[0]: 'MD'}).sort_values('MD')
sw = las['SW_QEPP'].clip(0, 1)
ugas = las['UGAS_COMBINER'].fillna(0)
uoil = las['UOIL_COMBINER'].fillna(0) if 'UOIL_COMBINER' in las.columns else 0 * ugas
las['S_GAS'] = ((1 - sw) * ugas / (ugas + uoil).replace(0, np.nan)).fillna(0)
p = pd.merge_asof(p, las[['MD', 'S_GAS']], on='MD', direction='nearest', tolerance=0.1)

p['Formation'] = p['Formation'].fillna('Not assigned')
p['error'] = p['VS_ACTUAL'] - p['VS_PRED_LASSO']            # + = model predicts Vs too LOW
p['vpvs_meas'] = p['VP_EDITED'] / p['VS_ACTUAL']
p['vpvs_pred'] = p['VP_EDITED'] / p['VS_PRED_LASSO']

# Table: error by formation (where the error becomes large, and what the rock is like there)
by_fm = p.groupby('Formation', sort=False).agg(
    top=('MD', 'min'), base=('MD', 'max'), rows=('MD', 'size'),
    mean_error=('error', 'mean'),
    RMSE=('error', lambda e: np.sqrt(np.mean(e ** 2))),
    sigma_error=('sigma', lambda s: np.nan),           # filled below
    gas_sat=('S_GAS', 'mean'),
    calcite=('CALCITE_COMBINER', 'mean'),
    vpvs_measured=('vpvs_meas', 'median'),
    vpvs_predicted=('vpvs_pred', 'median'),
)
by_fm['sigma_error'] = p.groupby('Formation', sort=False).apply(
    lambda g: np.sqrt(np.mean((g['sigma'] - g['sigma_actual']) ** 2)))
print(f"{name}: Lasso LOWO error by formation (error + = Vs predicted too low; sigma_error = RMSE of Poisson's ratio)")
display(by_fm.round(3))

# Figure: depth tracks
g = with_gaps(p)                                   # breaks lines at removed intervals
fig, axes = plt.subplots(1, 8, figsize=(22, 11), sharey=True,
                         gridspec_kw={'width_ratios': [1.3, 1, 1, 1, 1, 1.2, 1, 1]})

# 1. formations
fm_colors = plt.cm.tab20(np.linspace(0, 1, 20))
for k, (fm, row) in enumerate(by_fm.sort_values('top').iterrows()):
    axes[0].axhspan(row['top'], row['base'], color=fm_colors[k % 20], alpha=0.6)
    axes[0].text(0.5, (row['top'] + row['base']) / 2, fm, ha='center', va='center', fontsize=8, wrap=True)
axes[0].set_xticks([]); axes[0].set_xlabel('Formation')

axes[1].plot(g['GR_EDITED'], g['MD'], color='green', lw=0.7); axes[1].set_xlabel('GR (API)')
axes[2].fill_betweenx(g['MD'], 0, g['S_GAS'], color='red', alpha=0.5); axes[2].set_xlim(0, 1); axes[2].set_xlabel('Gas saturation')
axes[3].plot(g['CALCITE_COMBINER'], g['MD'], color='steelblue', lw=0.7); axes[3].set_xlabel('Calcite (fraction)')
axes[4].plot(g['vpvs_meas'], g['MD'], color='black', lw=0.7, label='measured')
axes[4].plot(g['vpvs_pred'], g['MD'], color='crimson', lw=0.7, label='predicted'); axes[4].set_xlabel('Vp/Vs'); axes[4].legend(fontsize=7)
axes[5].plot(g['VS_ACTUAL'], g['MD'], color='black', lw=0.7, label='measured')
axes[5].plot(g['VS_PRED_LASSO'], g['MD'], color='crimson', lw=0.7, label='predicted'); axes[5].set_xlabel('Vs (m/s)'); axes[5].legend(fontsize=7)
axes[6].fill_betweenx(g['MD'], 0, g['error'], where=g['error'] > 0, color='tab:blue', alpha=0.6, interpolate=True)
axes[6].fill_betweenx(g['MD'], 0, g['error'], where=g['error'] < 0, color='tab:orange', alpha=0.6, interpolate=True)
axes[6].axvline(0, color='black', lw=0.6); axes[6].set_xlabel('Vs error (m/s)\nblue = predicted too low')
axes[7].plot(g['sigma_actual'], g['MD'], color='black', lw=0.7, label='measured')
axes[7].plot(g['sigma'], g['MD'], color='crimson', lw=0.7, label='predicted'); axes[7].set_xlabel("Poisson's ratio"); axes[7].legend(fontsize=7)

for ax in axes[1:]:
    ax.grid(alpha=0.3)
axes[0].set_ylim(p['MD'].max() + 5, p['MD'].min() - 5)
axes[0].set_ylabel('MD (m)')
fig.suptitle(f'{name}: where the Lasso prediction fails (LOWO)', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / f'interpretation_{name}.png', dpi=PLOT_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# --- E–σ crossplots: measured and predicted in one plot, coloured by depth, GR, porosity and gas saturation ---
from matplotlib.colors import LinearSegmentedColormap

# Gas saturation from the LAS files (same method as estimate_sat_from_combiner.py)
sat_parts = []
for w in measured_wells:
    nm = w.replace('_with_formation', '')
    las = lasio.read(list(LAS_DIR.rglob(f"{nm}_LQC.las"))[0]).df().reset_index()
    las.columns = las.columns.str.upper()
    las = las.rename(columns={las.columns[0]: 'MD'})
    sw = las['SW_QEPP'].clip(0, 1)
    ugas = las['UGAS_COMBINER'].fillna(0)
    uoil = las['UOIL_COMBINER'].fillna(0) if 'UOIL_COMBINER' in las.columns else 0 * ugas
    las['S_GAS'] = ((1 - sw) * ugas / (ugas + uoil).replace(0, np.nan)).fillna(0)
    las['Well Name'] = w
    sat_parts.append(las[['Well Name', 'MD', 'S_GAS']])
sat = pd.concat(sat_parts).sort_values('MD')

pp = per_depth_df.merge(data_df[['Well Name', 'MD', 'GR_EDITED', 'PHIT_QEPP']], on=['Well Name', 'MD'], how='left')
pp = pd.merge_asof(pp.sort_values('MD'), sat, on='MD', by='Well Name', direction='nearest', tolerance=0.1)
pp['sigma_error'] = pp['sigma'] - pp['sigma_actual']         # + = predicted sigma too high
pp['E_error'] = pp['E_GPa'] - pp['E_GPa_actual']             # + = predicted E too high

# One-hue colour ramps (light -> dark), starting dark enough to stay visible on white
ramp = lambda hexes, nm: LinearSegmentedColormap.from_list(nm, hexes)
PANELS = [   # (column, title, colour-bar label, colour ramp)
    ('MD',        '(a) Coloured by depth',          'MD (m)',                    ramp(['#86b6ef', '#2a78d6', '#0d366b'], 'md')),
    ('GR_EDITED', '(b) Coloured by gamma ray',      'Gamma ray (API)',           ramp(['#9ee0c4', '#1baf7a', '#0b4d35'], 'gr')),
    ('PHIT_QEPP', '(c) Coloured by total porosity', 'Total porosity (fraction)', ramp(['#f6b48f', '#eb6834', '#7a2c0c'], 'phi')),
    ('S_GAS',     '(d) Coloured by gas saturation', 'Gas saturation (fraction)', ramp(['#f2a3a2', '#e34948', '#6e1414'], 'gas')),
]
GAS_MIN = 0.05                 # predicted points below this are drawn grey ("no gas")
C_MEAS = '#d6d5cf'             # measured-Vs points (light grey, behind)
C_NOGAS = '#8f8e89'            # predicted points with no gas

style = {'font.family': 'serif', 'font.size': 11, 'axes.edgecolor': '#c3c2b7', 'axes.linewidth': 0.8,
         'xtick.color': '#52514e', 'ytick.color': '#52514e', 'xtick.labelsize': 10, 'ytick.labelsize': 10}

for well in WELLS_OF_INTEREST:
    g = pp[pp['Well Name'] == well]
    well_label = well.replace('_with_formation', '')
    s_all = pd.concat([g['sigma'], g['sigma_actual']])
    e_all = pd.concat([g['E_GPa'], g['E_GPa_actual']])
    x_lim = (s_all.quantile(0.005) - 0.02, s_all.quantile(0.995) + 0.02)
    y_lim = (e_all.quantile(0.005) * 0.95, e_all.quantile(0.995) * 1.05)

    with plt.rc_context(style):
        fig, axes = plt.subplots(2, 2, figsize=(13, 11), sharex=True, sharey=True, constrained_layout=True)
        for ax, (var, title, cbar_label, cmap) in zip(axes.flat, PANELS):
            # measured: light grey cloud behind
            ax.scatter(g['sigma_actual'], g['E_GPa_actual'], s=5, color=C_MEAS, linewidths=0,
                       zorder=1, rasterized=True, label='from measured Vs')
            # predicted: coloured by the variable
            if var == 'S_GAS':
                no_gas = g[g['S_GAS'] < GAS_MIN]
                ax.scatter(no_gas['sigma'], no_gas['E_GPa'], s=5, color=C_NOGAS, alpha=0.6, linewidths=0,
                           zorder=2, rasterized=True, label='from predicted Vs, no gas')
                pts = g[g['S_GAS'] >= GAS_MIN].sort_values(var)
                vmin, vmax = GAS_MIN, 1.0
            else:
                pts = g.sort_values(var)
                vmin, vmax = g[var].quantile([0.02, 0.98])
            sc = ax.scatter(pts['sigma'], pts['E_GPa'], c=pts[var], cmap=cmap, vmin=vmin, vmax=vmax,
                            s=6, alpha=0.85, linewidths=0, zorder=3, rasterized=True,
                            label='from predicted Vs (colour)')
            cb = fig.colorbar(sc, ax=ax, pad=0.01, fraction=0.05, aspect=30)
            cb.set_label(cbar_label, fontsize=10); cb.outline.set_visible(False)
            cb.ax.yaxis.set_major_locator(plt.MaxNLocator(5))
            if var == 'MD':
                cb.ax.invert_yaxis()                      # shallow at the top
            ax.set_xlim(*x_lim); ax.set_ylim(*y_lim)
            ax.xaxis.set_major_locator(plt.MaxNLocator(5)); ax.yaxis.set_major_locator(plt.MaxNLocator(6))
            ax.grid(color='#e1e0d9', linewidth=0.6); ax.set_axisbelow(True)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            ax.set_title(title, loc='left', fontsize=12)
            leg = ax.legend(loc='upper right', fontsize=9, frameon=True, framealpha=0.9, edgecolor='none', markerscale=3)
            for h in leg.legend_handles: h.set_alpha(1)
        for ax in axes[1]: ax.set_xlabel("Poisson's ratio σ")
        for ax in axes[:, 0]: ax.set_ylabel("Young's modulus E (GPa)")
        fig.suptitle(f"{well_label}: Young's modulus vs Poisson's ratio (grey = from measured Vs, colour = from predicted Vs)",
                     fontsize=13, fontweight='bold')
        plt.savefig(PLOTS_DIR / f'E_sigma_coloured_{well_label}.png', dpi=PLOT_DPI, bbox_inches='tight')
        plt.show()

# Which variable organises the error? (Spearman correlation; 0 = no relation, NaN = variable constant, e.g. no gas)
rows = []
for well in WELLS_OF_INTEREST:
    g = pp[pp['Well Name'] == well]
    r = {'Well': well.replace('_with_formation', '')}
    for err, err_label in [('sigma_error', 'σ error'), ('E_error', 'E error')]:
        for var, _, label, _ in PANELS[1:]:
            r[f'{err_label} vs {label.split(" (")[0].lower()}'] = g[err].corr(g[var], method='spearman')
    r['mean gas saturation'] = g['S_GAS'].mean()
    rows.append(r)
err_corr = pd.DataFrame(rows).set_index('Well')
err_corr.round(2).to_csv(RESULTS_DIR / 'error_vs_GR_porosity_gas.csv')
print("Correlation of the σ and E errors (predicted minus measured) with GR, porosity and gas saturation")
display(err_corr.round(2))

In [ ]:
# --- Vp–Vs analysis: fitted lines for measured and predicted Vs vs published relationships ---
# Predicted Vs = Lasso LOWO predictions (each well predicted by a model trained on the other 4)
# Published relationships (velocities in km/s), brine-saturated rocks:
#   Mudrock line, Castagna et al. (1985):            Vs = 0.8621 Vp - 1.1724
#   Greenberg & Castagna (1992), sandstone:          Vs = 0.80416 Vp - 0.85588
#   Greenberg & Castagna (1992), shale:              Vs = 0.76969 Vp - 0.86735
#   Greenberg & Castagna (1992), limestone:          Vs = -0.05508 Vp^2 + 1.01677 Vp - 1.03049
REF = {
    'Mudrock line (Castagna et al., 1985)': (0.0, 0.8621, -1.1724),
    'Greenberg–Castagna sandstone':          (0.0, 0.80416, -0.85588),
    'Greenberg–Castagna shale':              (0.0, 0.76969, -0.86735),
    'Greenberg–Castagna limestone':          (-0.05508, 1.01677, -1.03049),
}
def ref_vs(name, vp_ms):                       # m/s in, m/s out
    a2, a1, a0 = REF[name]; v = vp_ms / 1000
    return 1000 * (a2 * v ** 2 + a1 * v + a0)

vv = per_depth_df.merge(data_df[['Well Name', 'MD', 'VCL_GEO_QEPP', 'CALCITE_COMBINER',
                                 'QUARTZ_COMBINER', 'ILLITE_COMBINER']], on=['Well Name', 'MD'], how='left')
calc = vv['CALCITE_COMBINER'] / vv[['QUARTZ_COMBINER', 'CALCITE_COMBINER', 'ILLITE_COMBINER']].sum(axis=1)
vv['Lithology'] = np.select(
    [calc >= 0.5, vv['VCL_GEO_QEPP'] >= 0.5, (vv['VCL_GEO_QEPP'] < 0.3) & (calc < 0.25)],
    ['Carbonate', 'Shale', 'Sand'], default='Mixed')

def fit_line(vp, vs):
    a, b = np.polyfit(vp, vs, 1)
    r2 = 1 - np.sum((vs - (a * vp + b)) ** 2) / np.sum((vs - vs.mean()) ** 2)
    return a, b, r2

# ---- Table 1: fitted lines, pooled and by lithology -------------------------------
REF_FOR = {'All rocks': 'Mudrock line (Castagna et al., 1985)', 'Sand': 'Greenberg–Castagna sandstone',
           'Shale': 'Greenberg–Castagna shale', 'Carbonate': 'Greenberg–Castagna limestone'}
rows = []
for group, ref_name in REF_FOR.items():
    d = vv if group == 'All rocks' else vv[vv['Lithology'] == group]
    for label, col in [('measured Vs', 'VS_ACTUAL'), ('predicted Vs (Lasso)', 'VS_PRED_LASSO')]:
        a, b, r2 = fit_line(d['VP_EDITED'].values, d[col].values)
        rows.append({'Rocks': group, 'Line': label, 'n': len(d), 'slope a': a, 'intercept b (m/s)': b, 'R² of fit': r2,
                     'Vs at Vp=3000': a * 3000 + b, 'Vs at Vp=4500': a * 4500 + b})
    a2, a1, a0 = REF[ref_name]
    rows.append({'Rocks': group, 'Line': ref_name, 'n': np.nan,
                 'slope a': a1 if a2 == 0 else np.nan, 'intercept b (m/s)': 1000 * a0 if a2 == 0 else np.nan,
                 'R² of fit': np.nan, 'Vs at Vp=3000': ref_vs(ref_name, 3000), 'Vs at Vp=4500': ref_vs(ref_name, 4500)})
fits = pd.DataFrame(rows)
fits.round(3).to_csv(RESULTS_DIR / 'vp_vs_fits_by_lithology.csv', index=False)
print("Table 1 - Vs = a·Vp + b fitted to measured and predicted Vs, with the published line for each rock type")
display(fits.round({'slope a': 3, 'intercept b (m/s)': 0, 'R² of fit': 3, 'Vs at Vp=3000': 0, 'Vs at Vp=4500': 0}))

# ---- Table 2: fitted lines per well --------------------------------------------------
rows = []
for well in WELLS_OF_INTEREST:
    d = vv[vv['Well Name'] == well]
    am, bm, rm = fit_line(d['VP_EDITED'].values, d['VS_ACTUAL'].values)
    ap, bp, rp = fit_line(d['VP_EDITED'].values, d['VS_PRED_LASSO'].values)
    rows.append({'Well': well.replace('_with_formation', ''), 'n': len(d),
                 'a measured': am, 'b measured (m/s)': bm, 'R² measured': rm,
                 'a predicted': ap, 'b predicted (m/s)': bp, 'R² predicted': rp})
fits_well = pd.DataFrame(rows).set_index('Well')
fits_well.round(3).to_csv(RESULTS_DIR / 'vp_vs_fits_by_well.csv')
print("\nTable 2 - fitted lines per well (mudrock line: a = 0.862, b = -1172 m/s)")
display(fits_well.round({'a measured': 3, 'b measured (m/s)': 0, 'R² measured': 3,
                         'a predicted': 3, 'b predicted (m/s)': 0, 'R² predicted': 3}))

# ---- Figures ---------------------------------------------------------------------------
style = {'font.family': 'serif', 'font.size': 10, 'axes.edgecolor': '#c3c2b7', 'axes.linewidth': 0.8,
         'xtick.color': '#52514e', 'ytick.color': '#52514e'}
C_MEAS, C_PRED, C_REF = '#9f9e98', '#2a78d6', '#e34948'
vp_lim = tuple(vv['VP_EDITED'].quantile([0.001, 0.999]))
vs_lim = (vv[['VS_ACTUAL', 'VS_PRED_LASSO']].min().min() * 0.95, vv[['VS_ACTUAL', 'VS_PRED_LASSO']].max().max() * 1.02)
x_line = np.linspace(*vp_lim, 100)

def style_ax(ax):
    ax.grid(color='#e1e0d9', linewidth=0.6); ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.set_xlim(*vp_lim); ax.set_ylim(*vs_lim)

def eq(a, b):
    return f'Vs = {a:.3f}·Vp {"+" if b >= 0 else "−"} {abs(b):.0f}'

with plt.rc_context(style):
    # Figure 1: all rocks, measured vs predicted, with the mudrock line
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True, constrained_layout=True)
    am, bm, _ = fit_line(vv['VP_EDITED'].values, vv['VS_ACTUAL'].values)
    ap, bp, _ = fit_line(vv['VP_EDITED'].values, vv['VS_PRED_LASSO'].values)
    mud = 'Mudrock line (Castagna et al., 1985)'
    for ax, col, colr, (a, b), title in [(axes[0], 'VS_ACTUAL', C_MEAS, (am, bm), '(a) Measured Vs'),
                                         (axes[1], 'VS_PRED_LASSO', C_PRED, (ap, bp), '(b) Predicted Vs (Lasso, LOWO)')]:
        ax.scatter(vv['VP_EDITED'], vv[col], s=3, color=colr, alpha=0.25, linewidths=0, rasterized=True)
        ax.plot(x_line, a * x_line + b, color='#0b0b0b', lw=2, label=f'Fitted: {eq(a, b)}')
        if col == 'VS_PRED_LASSO':
            ax.plot(x_line, am * x_line + bm, color='#52514e', lw=1.2, ls=':', label=f'Measured fit: {eq(am, bm)}')
        ax.plot(x_line, ref_vs(mud, x_line), color=C_REF, lw=1.5, ls='--', label=f'{mud}')
        ax.set_title(title, loc='left', fontweight='bold'); ax.set_xlabel('Vp (m/s)')
        ax.legend(loc='upper left', fontsize=8.5, frameon=False)
        style_ax(ax)
    axes[0].set_ylabel('Vs (m/s)')
    fig.suptitle('Vp–Vs relationship for the five measured wells', fontsize=13)
    plt.savefig(PLOTS_DIR / 'Vp_Vs_all_rocks.png', dpi=PLOT_DPI, bbox_inches='tight')
    plt.show()

    # Figure 2: by rock type, with the Greenberg–Castagna line for that rock type
    fig, axes = plt.subplots(1, 3, figsize=(17, 6), sharex=True, sharey=True, constrained_layout=True)
    for ax, group in zip(axes, ['Sand', 'Shale', 'Carbonate']):
        d = vv[vv['Lithology'] == group]
        ax.scatter(d['VP_EDITED'], d['VS_ACTUAL'], s=3, color=C_MEAS, alpha=0.3, linewidths=0, rasterized=True, label='measured Vs')
        ax.scatter(d['VP_EDITED'], d['VS_PRED_LASSO'], s=3, color=C_PRED, alpha=0.3, linewidths=0, rasterized=True, label='predicted Vs')
        am, bm, _ = fit_line(d['VP_EDITED'].values, d['VS_ACTUAL'].values)
        ap, bp, _ = fit_line(d['VP_EDITED'].values, d['VS_PRED_LASSO'].values)
        ax.plot(x_line, am * x_line + bm, color='#52514e', lw=2, label=f'Measured fit: {eq(am, bm)}')
        ax.plot(x_line, ap * x_line + bp, color='#0d366b', lw=2, label=f'Predicted fit: {eq(ap, bp)}')
        ref_name = REF_FOR[group]
        ax.plot(x_line, ref_vs(ref_name, x_line), color=C_REF, lw=1.5, ls='--', label=ref_name)
        if group == 'Shale':
            ax.plot(x_line, ref_vs(mud, x_line), color=C_REF, lw=1, ls=':', label=mud)
        ax.set_title(f'{group} (n = {len(d)})', loc='left', fontweight='bold'); ax.set_xlabel('Vp (m/s)')
        leg = ax.legend(loc='upper left', fontsize=8, frameon=False, markerscale=4)
        for h in leg.legend_handles: h.set_alpha(1)
        style_ax(ax)
    axes[0].set_ylabel('Vs (m/s)')
    fig.suptitle('Vp–Vs relationship by rock type', fontsize=13)
    plt.savefig(PLOTS_DIR / 'Vp_Vs_by_rock_type.png', dpi=PLOT_DPI, bbox_inches='tight')
    plt.show()

In [ ]:
# --- Interpretation: Egret_2, with a caliper panel (washout check) ---
WELL = 'Egret_2_with_formation'
name = WELL.replace('_with_formation', '')
BIT_SIZE_IN = 12.25            # Egret_2 is drilled at one bit size throughout (see Table 3/4 caliper check)
WASHOUT_IN = 2.0               # hole more than 2 in over bit size = washout
TOOL_BAD_IN = -1.0             # caliper reading this far under bit size = tool not working, not a real hole size

extra = ['Well Name', 'MD', 'Formation', 'GR_EDITED', 'CALCITE_COMBINER', 'CALI_EDITED']
p = (per_depth_df[per_depth_df['Well Name'] == WELL]
     .merge(data_df[extra], on=['Well Name', 'MD'], how='left')
     .sort_values('MD').reset_index(drop=True))

# Gas saturation from the LAS file (same method as estimate_sat_from_combiner.py)
las = lasio.read(list(LAS_DIR.rglob(f"{name}_LQC.las"))[0]).df().reset_index()
las.columns = las.columns.str.upper()
las = las.rename(columns={las.columns[0]: 'MD'}).sort_values('MD')
sw = las['SW_QEPP'].clip(0, 1)
ugas = las['UGAS_COMBINER'].fillna(0)
uoil = las['UOIL_COMBINER'].fillna(0) if 'UOIL_COMBINER' in las.columns else 0 * ugas
las['S_GAS'] = ((1 - sw) * ugas / (ugas + uoil).replace(0, np.nan)).fillna(0)
p = pd.merge_asof(p, las[['MD', 'S_GAS']], on='MD', direction='nearest', tolerance=0.1)

p['Formation'] = p['Formation'].fillna('Not assigned')
p['error'] = p['VS_ACTUAL'] - p['VS_PRED_LASSO']            # + = model predicts Vs too LOW
p['vpvs_meas'] = p['VP_EDITED'] / p['VS_ACTUAL']
p['vpvs_pred'] = p['VP_EDITED'] / p['VS_PRED_LASSO']

# Caliper diagnostics (same rule as Table 3/4)
p['excess'] = p['CALI_EDITED'] - BIT_SIZE_IN
p['cali_bad'] = p['excess'] < TOOL_BAD_IN                   # tool not working (reads well under bit size)
p['washout'] = ~p['cali_bad'] & (p['excess'] > WASHOUT_IN)

# Table: error by formation, with washout extent added
by_fm = p.groupby('Formation', sort=False).agg(
    top=('MD', 'min'), base=('MD', 'max'), rows=('MD', 'size'),
    mean_error=('error', 'mean'),
    RMSE=('error', lambda e: np.sqrt(np.mean(e ** 2))),
    gas_sat=('S_GAS', 'mean'),
    calcite=('CALCITE_COMBINER', 'mean'),
    vpvs_measured=('vpvs_meas', 'median'),
    vpvs_predicted=('vpvs_pred', 'median'),
    pct_washout=('washout', lambda w: 100 * w.mean()),
    pct_no_caliper=('cali_bad', lambda c: 100 * c.mean()),
)
by_fm['sigma_error'] = p.groupby('Formation', sort=False).apply(
    lambda g: np.sqrt(np.mean((g['sigma'] - g['sigma_actual']) ** 2)))
print(f"{name}: Lasso LOWO error by formation (error + = Vs predicted too low)")
display(by_fm.round(3))

wn = by_fm.loc[by_fm.index.str.contains('Withnell', case=False, na=False)]
if len(wn):
    print(f"\nWithnell Fm: mean error {wn['mean_error'].values[0]:+.0f} m/s, "
          f"{wn['pct_washout'].values[0]:.0f}% washout, {wn['pct_no_caliper'].values[0]:.0f}% caliper tool not working")

# Figure: depth tracks, with caliper added
g = with_gaps(p)
fig, axes = plt.subplots(1, 9, figsize=(24, 11), sharey=True,
                         gridspec_kw={'width_ratios': [1.3, 1, 1, 1, 1, 1, 1.2, 1, 1]})

fm_colors = plt.cm.tab20(np.linspace(0, 1, 20))
for k, (fm, row) in enumerate(by_fm.sort_values('top').iterrows()):
    axes[0].axhspan(row['top'], row['base'], color=fm_colors[k % 20], alpha=0.6)
    axes[0].text(0.5, (row['top'] + row['base']) / 2, fm, ha='center', va='center', fontsize=8, wrap=True)
axes[0].set_xticks([]); axes[0].set_title('Formation', fontsize=10)

axes[1].plot(g['CALI_EDITED'], g['MD'], color='saddlebrown', lw=0.7)
axes[1].axvline(BIT_SIZE_IN, color='black', lw=0.8, ls='--', label=f'Bit size ({BIT_SIZE_IN}")')
axes[1].fill_betweenx(g['MD'], BIT_SIZE_IN, g['CALI_EDITED'],
                       where=(g['CALI_EDITED'] - BIT_SIZE_IN) > WASHOUT_IN,
                       color='tab:red', alpha=0.4, interpolate=True, label='Washout')
no_cal = p[p['cali_bad']]
if len(no_cal):
    for _, r in no_cal.groupby((no_cal['MD'].diff() > 1).cumsum()):
        axes[1].axhspan(r['MD'].min(), r['MD'].max(), color='grey', alpha=0.25)
axes[1].set_title('Caliper (in)', fontsize=10); axes[1].legend(fontsize=6, loc='upper right')

axes[2].plot(g['GR_EDITED'], g['MD'], color='green', lw=0.7); axes[2].set_title('GR (API)', fontsize=10)
axes[3].fill_betweenx(g['MD'], 0, g['S_GAS'], color='red', alpha=0.5); axes[3].set_xlim(0, 1); axes[3].set_title('Gas saturation', fontsize=10)
axes[4].plot(g['CALCITE_COMBINER'], g['MD'], color='steelblue', lw=0.7); axes[4].set_title('Calcite (fraction)', fontsize=10)
axes[5].plot(g['vpvs_meas'], g['MD'], color='black', lw=0.7, label='Measured')
axes[5].plot(g['vpvs_pred'], g['MD'], color='crimson', lw=0.7, label='Predicted'); axes[5].set_title('Vp/Vs', fontsize=10); axes[5].legend(fontsize=7)
axes[6].plot(g['VS_ACTUAL'], g['MD'], color='black', lw=0.7, label='Measured')
axes[6].plot(g['VS_PRED_LASSO'], g['MD'], color='crimson', lw=0.7, label='Predicted'); axes[6].set_title('Vs (m/s)', fontsize=10); axes[6].legend(fontsize=7)
axes[7].fill_betweenx(g['MD'], 0, g['error'], where=g['error'] > 0, color='tab:blue', alpha=0.6, interpolate=True)
axes[7].fill_betweenx(g['MD'], 0, g['error'], where=g['error'] < 0, color='tab:orange', alpha=0.6, interpolate=True)
axes[7].axvline(0, color='black', lw=0.6); axes[7].set_title('Vs error (m/s)', fontsize=10)
axes[8].plot(g['sigma_actual'], g['MD'], color='black', lw=0.7, label='Measured')
axes[8].plot(g['sigma'], g['MD'], color='crimson', lw=0.7, label='Predicted'); axes[8].set_title("Poisson's ratio", fontsize=10); axes[8].legend(fontsize=7)

for ax in axes[1:]:
    ax.grid(alpha=0.3)
axes[0].set_ylim(p['MD'].max() + 5, p['MD'].min() - 5)
axes[0].set_ylabel('MD (m)')
fig.suptitle(f'{name}: Lasso prediction error by depth (LOWO)', fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR / f'interpretation_{name}.png', dpi=PLOT_DPI, bbox_inches='tight')
plt.show()

In [ ]:
# --- Error by formation across all 5 measured wells (for the cross-well interpretation) ---
MEASURED_WELLS = ['Egret_2', 'Lambert_8B', 'Nereus_1', 'Persephone_1', 'Seraph_1']

BIT_SIZE = {                      # well: [(from MD, to MD, bit size in inches)]
    'Egret_2':      [(0, 9999, 12.25)],
    'Lambert_8B':   [(0, 9999, 12.25)],
    'Nereus_1':     [(0, 9999, 8.5)],
    'Persephone_1': [(0, 9999, 12.25)],
    'Seraph_1':     [(0, 2960, 12.25), (2960, 9999, 8.5)],
}
WASHOUT_IN = 2.0
TOOL_BAD_IN = -1.0

extra = ['Well Name', 'MD', 'Formation', 'GR_EDITED', 'CALCITE_COMBINER', 'CALI_EDITED']
rows = []
for name in MEASURED_WELLS:
    WELL = f'{name}_with_formation'
    p = (per_depth_df[per_depth_df['Well Name'] == WELL]
         .merge(data_df[extra], on=['Well Name', 'MD'], how='left')
         .sort_values('MD').reset_index(drop=True))

    las = lasio.read(list(LAS_DIR.rglob(f"{name}_LQC.las"))[0]).df().reset_index()
    las.columns = las.columns.str.upper()
    las = las.rename(columns={las.columns[0]: 'MD'}).sort_values('MD')
    sw = las['SW_QEPP'].clip(0, 1)
    ugas = las['UGAS_COMBINER'].fillna(0)
    uoil = las['UOIL_COMBINER'].fillna(0) if 'UOIL_COMBINER' in las.columns else 0 * ugas
    las['S_GAS'] = ((1 - sw) * ugas / (ugas + uoil).replace(0, np.nan)).fillna(0)
    p = pd.merge_asof(p, las[['MD', 'S_GAS']], on='MD', direction='nearest', tolerance=0.1)

    p['Formation'] = p['Formation'].fillna('Not assigned')
    p['error'] = p['VS_ACTUAL'] - p['VS_PRED_LASSO']
    p['vpvs_meas'] = p['VP_EDITED'] / p['VS_ACTUAL']
    p['vpvs_pred'] = p['VP_EDITED'] / p['VS_PRED_LASSO']

    p['bit'] = np.nan
    for top, base, bit in BIT_SIZE[name]:
        p.loc[(p['MD'] >= top) & (p['MD'] < base), 'bit'] = bit
    p['excess'] = p['CALI_EDITED'] - p['bit']
    p['cali_bad'] = p['excess'] < TOOL_BAD_IN
    p['washout'] = ~p['cali_bad'] & (p['excess'] > WASHOUT_IN)

    by_fm = p.groupby('Formation', sort=False).agg(
        top=('MD', 'min'), base=('MD', 'max'), rows=('MD', 'size'),
        mean_error=('error', 'mean'),
        RMSE=('error', lambda e: np.sqrt(np.mean(e ** 2))),
        gas_sat=('S_GAS', 'mean'),
        calcite=('CALCITE_COMBINER', 'mean'),
        vpvs_measured=('vpvs_meas', 'median'),
        vpvs_predicted=('vpvs_pred', 'median'),
        pct_washout=('washout', lambda w: 100 * w.mean()),
        pct_no_caliper=('cali_bad', lambda c: 100 * c.mean()),
    ).reset_index()
    by_fm['sigma_error'] = by_fm['Formation'].map(
        p.groupby('Formation', sort=False).apply(lambda g: np.sqrt(np.mean((g['sigma'] - g['sigma_actual']) ** 2))))
    by_fm.insert(0, 'Well', name)
    rows.append(by_fm)

error_by_fm_all = pd.concat(rows, ignore_index=True).round(3)
error_by_fm_all.to_csv(RESULTS_DIR / 'error_by_formation_all_wells.csv', index=False)
print(f"Saved {len(error_by_fm_all)} well-formation rows to error_by_formation_all_wells.csv")
display(error_by_fm_all.sort_values('RMSE', ascending=False).head(15))

# Quick visual: which formations are the worst, and in how many wells do they show up?
summary = (error_by_fm_all[error_by_fm_all['rows'] >= 30]   # drop tiny/edge intervals
           .groupby('Formation')
           .agg(wells=('Well', 'nunique'), mean_RMSE=('RMSE', 'mean'), max_RMSE=('RMSE', 'max'))
           .query('wells >= 2')
           .sort_values('mean_RMSE', ascending=False))
print("\nFormations appearing in 2+ wells (rows >= 30), ranked by mean RMSE:")
display(summary.round(1))

fig, ax = plt.subplots(figsize=(9, max(3, 0.4 * len(summary))))
ax.barh(summary.index[::-1], summary['mean_RMSE'][::-1], color='#4C72B0')
ax.set_xlabel('Mean Lasso RMSE across wells (m/s)')
ax.set_title('Formations present in 2 or more measured wells, by mean prediction error')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rmse_by_formation_summary.png', dpi=PLOT_DPI, bbox_inches='tight')
plt.show()